# Stock Market Data — Exercises

In this project, you will download daily stock-market data and analyze it using basic Python.

You will work with:
- lists
- dictionaries
- loops
- conditions
- functions
- simple percentage calculations

No pandas is used in the exercises.

## 1. Downloading market data

The function below downloads daily historical prices from Yahoo Finance and returns them as a **list of dictionaries**.

You are **not expected to write or modify this function**. Run the cell once and use `fetch_market_history()` in the exercises below.

Each trading day is stored in this form:

```python
{
    "date": "2024-01-03",
    "open": 440.2,
    "high": 443.5,
    "low": 437.8,
    "close": 441.6,
    "adjusted_close": 416.9,
    "volume": 215430
}
```

In [ ]:
import requests
from datetime import datetime, timedelta, timezone
from functools import lru_cache


@lru_cache(maxsize=None)
def fetch_market_history(symbol, start_date, end_date):
    """Download daily historical market data for one ticker.

    Parameters
    ----------
    symbol : str
        Yahoo Finance ticker, for example "ZURN.SW" or "AAPL".
    start_date : str
        First date in YYYY-MM-DD format.
    end_date : str
        Last date in YYYY-MM-DD format.

    Returns
    -------
    list
        A list of dictionaries, one dictionary per trading day.
    """

    start = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end = end + timedelta(days=1)

    url = f"https://query2.finance.yahoo.com/v8/finance/chart/{symbol}"

    params = {
        "period1": int(start.timestamp()),
        "period2": int(end.timestamp()),
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
    }

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=10
    )
    response.raise_for_status()

    payload = response.json()

    error = payload.get("chart", {}).get("error")
    if error is not None:
        raise ValueError(error.get("description", "Could not download market data."))

    result = payload["chart"]["result"][0]

    timestamps = result.get("timestamp", [])
    quotes = result["indicators"]["quote"][0]

    adjusted = (
        result.get("indicators", {})
        .get("adjclose", [{}])[0]
        .get("adjclose", [])
    )

    history = []

    for i, timestamp in enumerate(timestamps):
        close_price = quotes["close"][i]

        if close_price is None:
            continue

        adjusted_close = close_price

        if i < len(adjusted) and adjusted[i] is not None:
            adjusted_close = adjusted[i]

        day = {
            "date": datetime.fromtimestamp(
                timestamp, tz=timezone.utc
            ).strftime("%Y-%m-%d"),
            "open": quotes["open"][i],
            "high": quotes["high"][i],
            "low": quotes["low"][i],
            "close": close_price,
            "adjusted_close": adjusted_close,
            "volume": quotes["volume"][i],
        }

        history.append(day)

    if len(history) == 0:
        raise ValueError(f"No price data found for {symbol}.")

    return history

### Exercise 1.1 — Download one stock

Download daily prices for **Zurich Insurance Group** (`ZURN.SW`) from January 1 to March 31, 2024.

Save the result as `zurich_prices`.

Then print:
1. the type of `zurich_prices`
2. the number of trading days returned
3. the first item
4. the last item

In [ ]:
zurich_prices = # Call fetch_market_history() here

# Print the requested information below

### Exercise 1.2 — Inspect one trading day

The items inside `zurich_prices` are dictionaries.

Using the **first trading day** in the list:

1. save the dictionary as `first_day`
2. print its date
3. print its opening price
4. print its closing price
5. print all dictionary keys

In [ ]:
first_day = # Enter code here

# Enter code here

### Exercise 1.3 — Calculate the daily price range

For the first trading day, calculate:

\[
	ext{daily range} = 	ext{high price} - 	ext{low price}
\]

Save the result as `first_day_range` and print it.

In [ ]:
first_day_range = # Enter code here

print(first_day_range)

## 2. Working with a list of price records

### Exercise 2.1 — Extract closing prices

Write a function called `get_closing_prices()`.

The function should:
1. take a list of daily price dictionaries as an argument
2. create an empty list
3. loop through the daily records
4. append each closing price to the new list
5. return the list

In [ ]:
def get_closing_prices(history):
    closing_prices = []

    # Add loop here

    return closing_prices


# Test
zurich_closes = get_closing_prices(zurich_prices)
print(zurich_closes[:5])

### Exercise 2.2 — Average closing price

Write a function called `average_price()`.

It should take a list of numbers and return their arithmetic mean.

Do not use NumPy.

In [ ]:
def average_price(prices):
    # Enter code here
    return


# Test
zurich_average = average_price(zurich_closes)
print(zurich_average)

### Exercise 2.3 — Highest and lowest close

Write a function called `price_bounds()` that takes a list of prices and returns:

```python
(lowest_price, highest_price)
```

You may use the built-in `min()` and `max()` functions.

In [ ]:
def price_bounds(prices):
    # Enter code here
    return


# Test
zurich_bounds = price_bounds(zurich_closes)
print(zurich_bounds)

## 3. Finding information inside the data

### Exercise 3.1 — Find a trading day

Write a function called `find_trading_day()`.

It should:
1. take `history` and `date` as arguments
2. loop through the daily records
3. return the dictionary whose `"date"` matches the requested date
4. return `None` if the date is not present

Remember that weekends and market holidays will not appear in the data.

In [ ]:
def find_trading_day(history, date):
    # Enter code here
    return


# Test cases
print(find_trading_day(zurich_prices, "2024-02-15"))
print(find_trading_day(zurich_prices, "2024-02-17"))

### Exercise 3.2 — Find one value

Write a function called `market_value()`.

It should accept:
- `history`
- `date`
- `field`

For example:

```python
market_value(zurich_prices, "2024-02-15", "close")
```

should return the closing price on that date.

Use your `find_trading_day()` function inside this function.

If the requested date is not found, return `None`.

In [ ]:
def market_value(history, date, field):
    # Enter code here
    return


# Test
print(market_value(zurich_prices, "2024-02-15", "close"))
print(market_value(zurich_prices, "2024-02-15", "volume"))

## 4. Price changes

### Exercise 4.1 — Percentage change

Write a function called `percent_change()` that takes a starting value and an ending value.

Use:

\[
rac{	ext{end} - 	ext{start}}{	ext{start}} 	imes 100
\]

The function should return the percentage change.

In [ ]:
def percent_change(start_value, end_value):
    # Enter code here
    return


# Test
print(percent_change(100, 110))
print(percent_change(100, 85))

### Exercise 4.2 — Return over the downloaded period

Use the **first** and **last** trading-day records in `zurich_prices`.

1. save the first closing price as `start_close`
2. save the last closing price as `end_close`
3. use `percent_change()` to calculate the return over the period
4. print the result

In [ ]:
start_close = # Enter code here
end_close = # Enter code here

period_return = # Enter code here

print(period_return)

### Exercise 4.3 — Turn it into a function

Write `stock_return()`.

It should take a list of daily records and return the percentage change from the first closing price to the last closing price.

Use your `percent_change()` function inside it.

In [ ]:
def stock_return(history):
    # Enter code here
    return


# Test
print(stock_return(zurich_prices))

## 5. Comparing several stocks

We will now compare three Swiss-listed companies over the same period:

- Zurich Insurance Group: `ZURN.SW`
- UBS Group: `UBSG.SW`
- Nestlé: `NESN.SW`

### Exercise 5.1 — Download several stocks

Create this list:

```python
swiss_symbols = ["ZURN.SW", "UBSG.SW", "NESN.SW"]
```

Then create an empty dictionary called `market_data`.

Loop through the ticker symbols. For each ticker:
1. download data from `"2024-01-01"` through `"2024-12-31"`
2. store the resulting list in `market_data`

The finished dictionary should have ticker symbols as keys and lists of daily records as values.

In [ ]:
swiss_symbols = ["ZURN.SW", "UBSG.SW", "NESN.SW"]

market_data = {}

# Add loop here

### Exercise 5.2 — Build a return dictionary

Create an empty dictionary called `stock_returns`.

Loop through `market_data`.

For each ticker:
1. calculate its return using `stock_return()`
2. store the result in `stock_returns`

The result should look approximately like:

```python
{
    "ZURN.SW": ...,
    "UBSG.SW": ...,
    "NESN.SW": ...
}
```

In [ ]:
stock_returns = {}

# Add loop here

print(stock_returns)

### Exercise 5.3 — Find the strongest return

Use a loop to find which ticker has the highest value in `stock_returns`.

Start with:

```python
best_ticker = None
best_return = None
```

Update these variables while looping through the dictionary.

At the end, print a sentence containing the ticker and its return.

Do not use `max(..., key=...)` for this exercise.

In [ ]:
best_ticker = None
best_return = None

# Add loop here

print(best_ticker, best_return)

## 6. One more market statistic

### Exercise 6.1 — Count up days and down days

Write a function called `count_market_days()`.

For each trading day:
- count an **up day** when `"close"` is greater than `"open"`
- count a **down day** when `"close"` is smaller than `"open"`
- ignore days where the two values are equal

Return:

```python
(up_days, down_days)
```

In [ ]:
def count_market_days(history):
    up_days = 0
    down_days = 0

    # Add loop here

    return up_days, down_days


# Test
print(count_market_days(zurich_prices))

### Exercise 6.2 — Compare the three stocks

Loop through `market_data` and print the number of up and down days for each ticker.

Example format:

```text
ZURN.SW: 130 up days, 121 down days
```

In [ ]:
# Enter code here

## Optional challenge

Write a function called `summary_report()` that accepts one ticker, a start date, and an end date.

It should download the data and return a dictionary containing:

```python
{
    "ticker": ...,
    "start": ...,
    "end": ...,
    "trading_days": ...,
    "average_close": ...,
    "lowest_close": ...,
    "highest_close": ...,
    "return_pct": ...
}
```

Reuse the functions you already wrote instead of repeating the calculations.

In [ ]:
def summary_report(ticker, start_date, end_date):
    # Enter code here
    return


# Test
print(summary_report("ZURN.SW", "2024-01-01", "2024-12-31"))
